<a href="https://colab.research.google.com/github/haidermalik68/Ai-image-checker/blob/main/03_evaluation_and_explanations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##3.0 Imports and Drive

In [ ]:
import io
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from google.colab import drive

drive.mount('/content/drive')

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)    # expect cuda

Mounted at /content/drive
Using: cuda


In [ ]:
## bring back images and tables
!cd /content && unzip -q /content/drive/MyDrive/ai_image_checker/data.zip
!cp /content/drive/MyDrive/ai_image_checker/*.csv /content/

test = pd.read_csv("/content/test.csv")
print(len(test))    # expect 12600

12600


### 3.0-c. Transforms and dataset class

In [ ]:
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

test_transform = transforms.Compose([
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

class ImageDataset(Dataset):
    def __init__(self, table, transform):
        self.table = table.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.table)

    def __getitem__(self, i):
        row = self.table.iloc[i]
        image = Image.open(row["file"]).convert("RGB")
        return self.transform(image), int(row["label"])

## 3.0-D.Rebuild the model and load your best weights

In [ ]:
save_folder = "/content/drive/MyDrive/ai_image_checker"

model = models.resnet18(weights=None)               # no ImageNet download, our weights come from the file
model.fc = nn.Linear(model.fc.in_features, 2)       # same shape as when we trained

model.load_state_dict(torch.load(save_folder + "/best_model.pt", map_location=device))
model = model.to(device)
model.eval()                                        # testing mode
print("model loaded")

model loaded


##3.1 Robustness: do the scores survive real-world damage?
* Photos shared on WhatsApp or social media are compressed and resized, so a detector that only works on clean files isn't useful.
* We re-test the same 12,600 test images after two kinds of damage: stronger JPEG compression (quality 50) and shrinking to 128×128 and enlarging back.
* A big drop means the model relied on fragile fine details.
This also helps check the shrink-amount shortcut idea from the README.
* Nothing is trained or changed here. We only measure.

## 3.1-A: Damage Functions

define two functions that take an image and return a damaged copy

In [ ]:
def degraded_jpeg(image,quality):
  buffer  = io.BytesIO()
  image.save(buffer,format="JPEG",quality=quality)
  buffer.seek(0)
  return Image.open(buffer)

def degraded_resize(image,size):
  small = image.resize((size,size))
  return small.resize((256,256))

## 3.1-B. Transforms with damage

In [ ]:
## two transforms that damage the image first and then apply te normal test steps
jpeg50_transform = transforms.Compose([
    transforms.Lambda(lambda img: degraded_jpeg(img,50)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

shrink128_transform = transforms.Compose([
    transforms.Lambda(lambda img: degraded_resize(img,128)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])




In [ ]:
def score_per_generator(transform, table):
    data = ImageDataset(table, transform)
    loader = DataLoader(data, batch_size=64, shuffle=False, num_workers=2)

    preds = []
    with torch.no_grad():
        for images, labels in loader:
            outputs = model(images.to(device))
            preds.extend(outputs.argmax(1).cpu().tolist())

    result = table.copy().reset_index(drop=True)
    result["correct"] = (pd.Series(preds) == result["label"])
    return result.groupby("generator")["correct"].mean().round(3)

In [ ]:
print(test["generator"].value_counts())

generator
midjourney    3000
dalle3        3000
sd3           3000
real          1800
sd21           900
sdxl           900
Name: count, dtype: int64


In [ ]:
##take a small sample of test dataset
small_test = test.groupby("generator").sample(n=900,random_state=0)
print(small_test["generator"].value_counts())

generator
dalle3        900
midjourney    900
real          900
sd21          900
sd3           900
sdxl          900
Name: count, dtype: int64


In [ ]:
##3.1-D.  Run all three and compare
## scores the original,shrink128,jpeg50, making a table and compare them

comparison = pd.DataFrame({
    "original": score_per_generator(test_transform,small_test),
    "shrink128": score_per_generator(shrink128_transform,small_test),
    "jpeg50": score_per_generator(jpeg50_transform,small_test),
})
print(comparison)
comparison.to_csv(save_folder + "/robustness_results.csv")

            original  shrink128  jpeg50
generator                              
dalle3         0.076      0.388   0.030
midjourney     0.326      0.304   0.140
real           0.964      0.829   0.981
sd21           0.846      0.929   0.646
sd3            0.178      0.361   0.094
sdxl           0.956      0.970   0.838


In [ ]:
import os, time

old = pd.read_csv(save_folder + "/test_results.csv")
same_images = old[old["file"].isin(small_test["file"])]
print(same_images.groupby("generator")["correct"].mean().round(3))

for name in ["best_model.pt", "last_model.pt", "test_results.csv"]:
    print(name, time.ctime(os.path.getmtime(save_folder + "/" + name)))

generator
dalle3        0.111
midjourney    0.598
real          0.882
sd21          0.927
sd3           0.378
sdxl          0.971
Name: correct, dtype: float64
best_model.pt Sun Nov 22 17:44:50 2026
last_model.pt Sun Nov 22 17:45:08 2026
test_results.csv Sun Nov 22 08:46:00 2026


In [16]:
# 3.1-G. Keep the named copy of model
!cp {save_folder}/best_model.pt {save_folder}/run_B_best_model.pt

In [17]:
# 3.1-H: Score Run B on the full test set
# scores all 12,600 test images with the plain transform and saves the table under a run-specific name.

run_b = score_per_generator(test_transform,test)
print(run_b)
run_b.to_csv(save_folder+"/run_B_full_test.csv")



generator
dalle3        0.080
midjourney    0.309
real          0.958
sd21          0.846
sd3           0.186
sdxl          0.956
Name: correct, dtype: float64


In [18]:
##3.1-I. Full robustness for run_B
run_b_robust = pd.DataFrame({
    "original": run_b,
    "jpeg50": score_per_generator(jpeg50_transform, test),
    "shrink128": score_per_generator(shrink128_transform, test),
})
print(run_b_robust)
run_b_robust.to_csv(save_folder + "/run_B_robustness_full.csv")

            original  jpeg50  shrink128
generator                              
dalle3         0.080   0.030      0.392
midjourney     0.309   0.130      0.280
real           0.958   0.973      0.820
sd21           0.846   0.646      0.929
sd3            0.186   0.094      0.352
sdxl           0.956   0.838      0.970
